# Phase 5 — Legal Statute Identification — Model 6 — LoRA

Run this notebook from top to bottom in a fresh kernel with the matching foundation checkpoint and tokenizer. The experiment is already selected in the configuration cell.

**Source:** `phase5/capstone-p5-all-models-all-methods.ipynb` at commit `7e7c8a9f1605c5f9836a3cb2cb7c28004343f5f3`. Model/training/evaluation/submission implementations are copied unchanged; only configuration selections and output directories differ. Saved outputs and execution counts were cleared during organization.

**Outputs:** use a separate Kaggle session or working directory for each run. The original submission filenames are preserved, so some ZIP files at the working-directory root can be overwritten by another run.

## Original implementation notes

# Phase 5 — Legal Statute Identification (LSI)
## Own pre-trained models (Model 4 / Model 6 / Model 7) + Full / LoRA / LoRMA | hierarchical **multi-label** classifier

Same methodology as the InLegalBERT Phase 5 notebook: one case = one sample, several windows per case are encoded, window vectors are aggregated (`attention` / `mean` / `label_attention`), and a sigmoid head predicts statutes (binary relevance). Threshold selection (global vs per-label, CV-checked on dev), loss options (`bce` / `focal` / `asl`), submission format are unchanged.

**What changed:** no InLegalBERT anywhere. Pick the backbone with `ARCH` in Section 2:

| `ARCH` | Backbone | Tokenizer | Window vector |
|---|---|---|---|
| `model4` | EfficientLegalLM 768d / 16 layers, GQA 12Q/2KV, window 256 | BPE `tokenizer.json` | masked mean of tokens |
| `model6` | EfficientLegalLM 1024d / 20 layers, GQA 16Q/4KV, window 512 | BPE `tokenizer.json` | masked mean of tokens |
| `model7` | BERT-style 12 layers, 768d, uncased WordPiece 30,522 | `vocab.txt` | masked mean of tokens |

Models 4/6 are causal LMs; at fine-tune time they run **bidirectionally** (as in the Phase 3 notebook). Model 7 uses the whole window as bidirectional prefix (segment 0).

**Methods:** `full` (all weights), `lora` (additive low-rank), `lorma` (multiplicative low-rank) and **`kd`** (knowledge distillation): the fully fine-tuned model is the *teacher*; a half-depth *student* initialised from evenly spaced pre-trained layers learns from `(1-KD_ALPHA)*task loss + KD_ALPHA*T^2*BCE(student/T, sigmoid(teacher/T))`. The student makes the submission. Teacher train-set logits are cached to `.npy` so a re-run does not repeat that inference.

**Attach as Kaggle inputs:** the checkpoint dataset (`best.pt`) of the chosen model and its tokenizer dataset.

## 1. Install and imports

In [ ]:
%pip install -q "transformers>=4.44,<5" "huggingface_hub>=0.24,<2" "scikit-learn>=1.3" "safetensors>=0.4"

In [ ]:
import os, json, math, random, re, gc, time, copy, zipfile, glob
from pathlib import Path
from collections import Counter
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import PreTrainedTokenizerFast, BertTokenizerFast, AutoTokenizer, BertConfig, BertModel, get_linear_schedule_with_warmup
from huggingface_hub import hf_hub_download
from sklearn.metrics import f1_score
print('torch',torch.__version__,'cuda',torch.cuda.is_available())

## 2. Configuration — choose architecture, method and representation

In [ ]:
ARCH = 'model6'                # <<< CHOOSE: 'model4' | 'model6' | 'model7'
RUN_METHODS = ['lora']         # any of 'full' | 'lora' | 'lorma' | 'kd' (run sequentially)
REPRESENTATION = 'head_tail'   # 'uniform' | 'head_tail' | 'sections'
POOLING = 'label_attention'    # 'attention' | 'mean' | 'label_attention'
LOSS = 'bce'                   # 'bce' (pos_weight) | 'focal' | 'asl'
THRESHOLD_MODE = 'auto'        # 'global' | 'per_label' | 'auto'
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
P5 = 'Phase_5_LSI'
HF_TOKEN = os.environ.get('HF_TOKEN')

CKPT_PATH = None               # None = auto-detect /kaggle/input/**/best.pt
TOK_DIR = None                 # None = auto-detect tokenizer dataset
INIT_CKPT = None               # optional: a best_<method>_lsi.pt from an earlier run of THIS notebook (warm start)
CKPT_KEY_PREFIX = ''           # model7 only: extra prefix to strip from checkpoint keys (after _orig_mod./module./bert.)

# ---- architecture table ----
assert ARCH in ('model4','model6','model7'), "ARCH must be 'model4', 'model6' or 'model7'"
if ARCH == 'model4':
    KIND='eff'; D_MODEL,N_LAYERS,N_HEADS,N_KV_HEADS,D_FF,WINDOW_SIZE=768,16,12,2,2048,256
elif ARCH == 'model6':
    KIND='eff'; D_MODEL,N_LAYERS,N_HEADS,N_KV_HEADS,D_FF,WINDOW_SIZE=1024,20,16,4,2432,512
else:
    KIND='bert'
if KIND=='eff':
    HEAD_DIM=D_MODEL//N_HEADS; MAX_POS=1024; ROPE_BASE=10000.0
    N_SPECIAL=1                 # windows are  ids + [SEP]
else:
    M7_CFG=dict(vocab_size=30522,hidden_size=768,num_hidden_layers=12,num_attention_heads=12,intermediate_size=3072,
                hidden_act='gelu',max_position_embeddings=512,type_vocab_size=2,layer_norm_eps=1e-12,
                hidden_dropout_prob=0.1,attention_probs_dropout_prob=0.1,position_embedding_type='absolute')
    D_MODEL=M7_CFG['hidden_size']; MAX_POS=M7_CFG['max_position_embeddings']
    N_SPECIAL=2                 # windows are  [CLS] + ids + [SEP]
HID=D_MODEL
NL_FULL = N_LAYERS if KIND=='eff' else M7_CFG['num_hidden_layers']

OUTPUT_DIR = Path('/kaggle/working/lsi_p5_model6_lora' if Path('/kaggle/working').exists() else '/mnt/data/lsi_p5_model6_lora')
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)

SEED=42; SMOKE=False; MAX_LEN=384; CHUNKS=6; STRIDE=256
BATCH_SIZE=1; GRAD_ACCUM=8; INFER_BATCH=1
EPOCHS=5; PATIENCE=3; HEAD_LR=5e-4; FULL_LR=5e-5; ADAPTER_LR=3e-4
WEIGHT_DECAY=0.01; WARMUP_FRAC=0.03
RANK=8; ALPHA=16.0; DROPOUT=0.1; GRAD_CHECKPOINT=True
NUM_WORKERS=0
# --- knowledge distillation (method 'kd') ---
KD_TEACHER_CKPT = None         # best_*_lsi.pt of a FULL fine-tune of this ARCH -> skips teacher training
KD_TEACHER_LR   = FULL_LR
KD_TEMP         = 2.0
KD_ALPHA        = 0.5
KD_STUDENT_LAYERS = NL_FULL // 2
KD_LAYER_MAP    = [int(round(x)) for x in np.linspace(0, NL_FULL - 1, KD_STUDENT_LAYERS)]
assert len(set(KD_LAYER_MAP)) == KD_STUDENT_LAYERS
BIDIR_WINDOW = WINDOW_SIZE if KIND=='eff' else None   # sliding-window width of the "window" layers at fine-tune time

# --- imbalance / rare-label handling ---
POS_WEIGHT_POWER=0.5; POS_WEIGHT_MAX=30.0
FOCAL_GAMMA=2.0
ASL_GAMMA_POS=0.0; ASL_GAMMA_NEG=4.0; ASL_CLIP=0.05
# --- thresholding ---
MIN_SUPPORT_PL=3; SHRINK_PL=0.7; MIN_PRED=1

assert all(m in ('full','lora','lorma','kd') for m in RUN_METHODS)
assert REPRESENTATION in ('uniform','head_tail','sections')
assert POOLING in ('attention','mean','label_attention')
assert LOSS in ('bce','focal','asl') and THRESHOLD_MODE in ('global','per_label','auto')
assert MAX_LEN <= MAX_POS
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP=DEVICE.type=='cuda'; AMP_DTYPE=torch.float16
print(f'ARCH={ARCH} kind={KIND} hidden={D_MODEL} | device={DEVICE} | out={OUTPUT_DIR}')

## 2b. Locate checkpoint and tokenizer, load tokenizer

In [ ]:
def _find(patterns, prefer=()):
    hits = sorted({h for p in patterns for h in glob.glob(p, recursive=True)})
    for key in prefer:
        pref = [h for h in hits if key in h.lower()]
        if pref: return pref[0], hits
    return (hits[0] if hits else None), hits

_PREF = {'model4':('m4','model4','model_4'),'model6':('m6','model6','model_6'),'model7':('m7','model7','model_7')}[ARCH]
if CKPT_PATH is None:
    CKPT_PATH, hits = _find(['/kaggle/input/**/best.pt'], prefer=_PREF+('capstone',))
    print('best.pt candidates:', hits)
if TOK_DIR is None:
    pats = ['/kaggle/input/**/tokenizer.json'] if KIND=='eff' else ['/kaggle/input/**/vocab.txt','/kaggle/input/**/tokenizer.json']
    tj, hits = _find(pats, prefer=_PREF+('legal','tokenizer'))
    TOK_DIR = os.path.dirname(tj) if tj else None
    print('tokenizer candidates:', hits)
assert CKPT_PATH and os.path.isfile(CKPT_PATH), 'checkpoint not found - attach the dataset or set CKPT_PATH'
assert TOK_DIR and os.path.isdir(TOK_DIR), 'tokenizer not found - attach the dataset or set TOK_DIR'
print('using checkpoint:', CKPT_PATH); print('using tokenizer :', TOK_DIR)

if KIND=='eff':
    tokenizer = PreTrainedTokenizerFast.from_pretrained(TOK_DIR)
else:
    try: tokenizer = BertTokenizerFast.from_pretrained(TOK_DIR, do_lower_case=True)
    except Exception as e:
        print('BertTokenizerFast failed (',e,') - falling back to AutoTokenizer'); tokenizer = AutoTokenizer.from_pretrained(TOK_DIR)
tokenizer.model_max_length = 10**9
VOCAB_SIZE = len(tokenizer)
if KIND=='bert':
    assert VOCAB_SIZE == M7_CFG['vocab_size'], f'tokenizer has {VOCAB_SIZE} entries, Model 7 expects {M7_CFG["vocab_size"]}'
    assert None not in (tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id)
    PAD_ID, SEP_ID, CLS_ID = tokenizer.pad_token_id, tokenizer.sep_token_id, tokenizer.cls_token_id
else:
    PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    SEP_ID = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else PAD_ID
    CLS_ID = None
print(f'VOCAB_SIZE={VOCAB_SIZE} PAD={PAD_ID} SEP={SEP_ID} CLS={CLS_ID} fast={tokenizer.is_fast}')

def wrap_window(ids):
    """Add the architecture's special tokens around a window of raw token ids."""
    return ([CLS_ID]+list(ids)+[SEP_ID]) if KIND=='bert' else (list(ids)+[SEP_ID])

## 3. Download Phase 5 data, validate schema, build the label space

In [ ]:
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF")
    except Exception:
        pass

def fetch(rel):
    return hf_hub_download(repo_id=HF_DATASET_REPO, repo_type="dataset", filename=rel, token=HF_TOKEN)

LABEL_KEYS=("labels","label","statutes","statute_ids","target")

def parse_labels(y, rid):
    if y is None: return []
    if isinstance(y,(int,np.integer)): y=[y]
    if isinstance(y,str):
        y=[t for t in re.split(r'[,\s;]+',y.strip('[] ')) if t]
    out=set()
    for v in y:
        try: out.add(int(v))
        except Exception: raise ValueError(f'Non-integer statute id {v!r} in {rid}')
    return sorted(out)

def normalize(raw, labeled, split_name="unknown"):
    if isinstance(raw, dict):
        if isinstance(raw.get("data"), list): raw = raw["data"]
        elif isinstance(raw.get("examples"), list): raw = raw["examples"]
        else: raw=[dict(v,id=k) if isinstance(v,dict) else {"id":k,"text":v} for k,v in raw.items()]
    assert isinstance(raw, list), "Expected list of records or id-indexed dictionary"
    out=[]
    for row in raw:
        assert isinstance(row,dict) and "id" in row and "text" in row, f"Bad row: {str(row)[:200]}"
        text=row["text"]
        if isinstance(text,list): text="\n".join(map(str,text))
        assert isinstance(text,str), f'Unexpected text for {row["id"]}'
        r={"id":str(row["id"]),"text":text}
        if labeled:
            keys=[k for k in LABEL_KEYS if k in row]
            assert keys, f'Missing labels for {r["id"]}, available keys {list(row)}'
            r["labels"]=parse_labels(row[keys[0]], r["id"])
        out.append(r)
    counts=Counter(r["id"] for r in out); dups={k:v for k,v in counts.items() if v>1}
    print(f"\n{'='*55}\nSplit: {split_name} | records {len(out):,} | unique ids {len(counts):,} | duplicated ids {len(dups):,}")
    merged={}; seen={}; label_union_changes=0
    for r in out:
        i=r["id"]
        if i not in merged: merged[i]=dict(r); seen[i]={r["text"]}; continue
        if r["text"] not in seen[i] and r["text"].strip():
            merged[i]["text"]+="\n\n"+r["text"]; seen[i].add(r["text"])
        if labeled and r["labels"]!=merged[i]["labels"]:
            merged[i]["labels"]=sorted(set(merged[i]["labels"])|set(r["labels"])); label_union_changes+=1
    out=list(merged.values())
    if labeled: print("Duplicate-id records whose label sets differed (unioned):",label_union_changes)
    print(f"Final cases: {len(out):,}\n{'='*55}")
    assert len({r['id'] for r in out})==len(out)
    return out

def read_split(name):
    folder="Test" if name=="test" else "Train"
    with open(fetch(f"{P5}/{folder}/{name}.json"),encoding="utf-8") as f: raw=json.load(f)
    return normalize(raw,labeled=(name!="test"),split_name=name)

train=read_split("train"); dev=read_split("dev"); test=read_split("test")
overlap={x['id'] for x in train}&{x['id'] for x in dev}
assert not overlap, f"Train and dev share {len(overlap)} case IDs (leakage)"

tr_ids=sorted({l for r in train for l in r['labels']}); dv_ids=sorted({l for r in dev for l in r['labels']})
LABEL_IDS=sorted(set(tr_ids)|set(dv_ids)); L2I={l:i for i,l in enumerate(LABEL_IDS)}; NUM_LABELS=len(LABEL_IDS)
print(f"Statute ids: train {len(tr_ids)} | dev {len(dv_ids)} | union {NUM_LABELS} | in dev but never in train: {len(set(dv_ids)-set(tr_ids))}")
assert NUM_LABELS>1
if not all(len(r['labels'])>0 for r in train+dev): print("WARNING: some train/dev docs have an empty label set")

def to_matrix(records):
    Y=np.zeros((len(records),NUM_LABELS),dtype=np.float32)
    for i,r in enumerate(records):
        for l in r['labels']: Y[i,L2I[l]]=1
    return Y

for name,arr in [("train",train),("dev",dev),("test",test)]:
    lengths=[len(r["text"]) for r in arr]
    print(f"{name.upper():5s} cases {len(arr):,} | median chars {int(np.median(lengths)):,} | mean {int(np.mean(lengths)):,} | max {max(lengths):,}")

if SMOKE:
    train,dev,test=train[:24],dev[:12],test[:12]; EPOCHS=1; print("SMOKE MODE ENABLED")
assert len(train) and len(dev) and len(test)
Y_train=to_matrix(train); Y_dev=to_matrix(dev)
print("Schema example:",{"id":train[0]["id"],"text_chars":len(train[0]["text"]),"labels":train[0]["labels"]})

## 3b. Label statistics

In [ ]:
pos=Y_train.sum(0); card=Y_train.sum(1)
print(f"Labels per doc (train): mean {card.mean():.2f} | median {np.median(card):.0f} | max {int(card.max())} | docs with 0 labels {(card==0).sum()}")
print(f"Train positives per label: median {np.median(pos):.0f} | min {int(pos.min())} | max {int(pos.max())}")
for k in (0,1,2,5,10,20): print(f"  labels with <= {k:2d} train positives: {(pos<=k).sum()}")
top=np.argsort(-pos)[:10]; print("Most frequent:",[(LABEL_IDS[i],int(pos[i])) for i in top])
co=Y_train.T@Y_train; np.fill_diagonal(co,0)
pairs=[(LABEL_IDS[i],LABEL_IDS[j],int(co[i,j])) for i,j in zip(*np.triu_indices(NUM_LABELS,1)) if co[i,j]>0]
pairs=sorted(pairs,key=lambda t:-t[2])[:10]; print("Top co-occurring pairs:",pairs)
dpos=Y_dev.sum(0); print(f"Dev: {(dpos>0).sum()} / {NUM_LABELS} labels have >=1 positive; {(dpos>=MIN_SUPPORT_PL).sum()} have >= {MIN_SUPPORT_PL}")

## 4. Document window selection
Tokenize the whole document once (no truncation), keep at most `CHUNKS` windows of `MAX_LEN` tokens (incl. special tokens of the chosen architecture).

In [ ]:
WINDOW_TOKENS=MAX_LEN-N_SPECIAL
assert 16<STRIDE<=WINDOW_TOKENS
HEADER_RE=re.compile(r'(?im)^\s*(?:\d+[.)]\s*)?(facts?|background|brief facts|issues?|arguments?|submissions?|reasoning|analysis|discussion|findings?|conclusion|final order|order|judg(?:e)?ment)\s*[:.\-]?\s*$')

def make_starts(n, mode):
    if n<=WINDOW_TOKENS: return [0]
    max_start=n-WINDOW_TOKENS
    all_starts=list(range(0,max_start+1,STRIDE))
    if all_starts[-1]!=max_start: all_starts.append(max_start)
    if len(all_starts)<=CHUNKS: return all_starts
    inds=np.linspace(0,len(all_starts)-1,CHUNKS).round().astype(int)
    return [all_starts[i] for i in inds]

def windows_for(text, mode=REPRESENTATION):
    ids=tokenizer(text,add_special_tokens=False,truncation=False)['input_ids']
    if not ids: ids=[tokenizer.unk_token_id if tokenizer.unk_token_id is not None else PAD_ID]
    if mode=='sections':
        anchors=[]
        for match in list(HEADER_RE.finditer(text))[:40]:
            offset=len(tokenizer(text[:match.start()],add_special_tokens=False)['input_ids'])
            anchors.append(max(0,min(offset,len(ids)-WINDOW_TOKENS)))
        desired=[0,max(0,len(ids)-WINDOW_TOKENS)]+anchors+make_starts(len(ids),'head_tail')
        chosen=[]
        for v in desired:
            if not chosen or all(abs(v-a)>=WINDOW_TOKENS//3 for a in chosen): chosen.append(v)
            if len(chosen)>=CHUNKS: break
        starts=sorted(chosen)
    else: starts=make_starts(len(ids),mode)
    chunks=[wrap_window(ids[st:st+WINDOW_TOKENS]) for st in starts[:CHUNKS]]
    return chunks or [wrap_window(ids[:WINDOW_TOKENS])]

class CaseDataset(Dataset):
    def __init__(self,records,mode,Y=None):
        self.rows=records; self.Y=Y
        self.data=[windows_for(r['text'],mode) for r in tqdm(records,desc=f'Windowize {mode}')]
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        y=self.Y[i] if self.Y is not None else np.zeros(NUM_LABELS,dtype=np.float32)
        return self.data[i], y, self.rows[i]['id']

def collate(batch):
    B=len(batch); C=max(len(z[0]) for z in batch); L=max(len(w) for z in batch for w in z[0])
    ids=torch.full((B,C,L),PAD_ID,dtype=torch.long)
    mask=torch.zeros((B,C,L),dtype=torch.long); cm=torch.zeros((B,C),dtype=torch.bool)
    for b,(wins,_,_) in enumerate(batch):
        for c,w in enumerate(wins): ids[b,c,:len(w)]=torch.tensor(w); mask[b,c,:len(w)]=1; cm[b,c]=True
    ys=torch.from_numpy(np.stack([z[1] for z in batch])).float()
    return ids,mask,cm,ys,[z[2] for z in batch]

train_ds=CaseDataset(train,REPRESENTATION,Y_train); dev_ds=CaseDataset(dev,REPRESENTATION,Y_dev); test_ds=CaseDataset(test,REPRESENTATION)
print('Example # chunks:',len(train_ds.data[0]),'first chunk tokens',len(train_ds.data[0][0]))
print('Example window start:',tokenizer.decode(train_ds.data[0][0][:40]))

## 5. Own pre-trained backbones (Model 4 / 6 = EfficientLegalLM, Model 7 = BERT-style)

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).type_as(x) * self.weight

def precompute_rope_freqs(dim, max_seq_len, base=10000.0):
    freqs = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, freqs_cos, freqs_sin):
    x_r = x.reshape(*x.shape[:-1], -1, 2)
    x0, x1 = x_r[..., 0], x_r[..., 1]
    seq_len = x.shape[2]
    cos = freqs_cos[:seq_len].unsqueeze(0).unsqueeze(0).to(x.dtype)
    sin = freqs_sin[:seq_len].unsqueeze(0).unsqueeze(0).to(x.dtype)
    return torch.stack([x0 * cos - x1 * sin, x0 * sin + x1 * cos], dim=-1).reshape_as(x)

class SwiGLU_FFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.w_gate = nn.Linear(d_model, d_ff, bias=False); self.w_up = nn.Linear(d_model, d_ff, bias=False)
        self.w_down = nn.Linear(d_ff, d_model, bias=False); self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))

def repeat_kv(x, n_rep):
    if n_rep == 1: return x
    B, H, T, D = x.shape
    return x[:, :, None, :, :].expand(B, H, n_rep, T, D).reshape(B, H * n_rep, T, D)

class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, head_dim, dropout=0.0, use_window=False, window_size=256):
        super().__init__()
        self.n_heads = n_heads; self.n_kv_heads = n_kv_heads; self.head_dim = head_dim; self.n_rep = n_heads // n_kv_heads
        self.use_window = use_window; self.window_size = window_size; self.attn_p = dropout
        self.q_proj = nn.Linear(d_model, n_heads * head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * head_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * head_dim, d_model, bias=False)
        self.resid_dropout = nn.Dropout(dropout)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, head_dim, d_ff, dropout=0.0, use_window=False, window_size=256):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQAttention(d_model, n_heads, n_kv_heads, head_dim, dropout, use_window, window_size)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU_FFN(d_model, d_ff, dropout)

class EfficientLegalLM(nn.Module):
    """Phase-1 LM (Model 4 / 6). Only the modules needed for loading + bidirectional encoding are defined here."""
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, head_dim, d_ff, max_seq_len, rope_base=10000.0, dropout=0.0, window_size=256):
        super().__init__()
        self.d_model = d_model; self.n_layers = n_layers; self.vocab_size = vocab_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([TransformerBlock(d_model, n_heads, n_kv_heads, head_dim, d_ff, dropout,
                                                      use_window=(i % 2 == 0), window_size=window_size) for i in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
        fc, fs = precompute_rope_freqs(head_dim, max_seq_len, rope_base)
        self.register_buffer("freqs_cos", fc, persistent=False); self.register_buffer("freqs_sin", fs, persistent=False)

def build_masks(T, pad_mask, window):
    """Bidirectional boolean masks (True = may attend), (B,1,T,T): one for window layers, one for full layers."""
    r = torch.arange(T, device=pad_mask.device).unsqueeze(1); c = torch.arange(T, device=pad_mask.device).unsqueeze(0)
    full = torch.ones(T, T, dtype=torch.bool, device=pad_mask.device); win = (c - r).abs() < window
    key = pad_mask[:, None, None, :]
    return full[None, None] & key, win[None, None] & key

def block_forward(blk, x, cos, sin, mask):
    a = blk.attn; h = blk.attn_norm(x); B, T, _ = h.shape
    q = a.q_proj(h).view(B, T, a.n_heads, a.head_dim).transpose(1, 2)
    k = a.k_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    v = a.v_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    q = apply_rope(q, cos, sin); k = apply_rope(k, cos, sin)
    k = repeat_kv(k, a.n_rep); v = repeat_kv(v, a.n_rep)
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
    out = out.transpose(1, 2).contiguous().view(B, T, -1)
    x = x + a.out_proj(out)
    return x + blk.ffn(blk.ffn_norm(x))

def _norm_key(k):
    for pre in ("_orig_mod.", "module."):
        while k.startswith(pre): k = k[len(pre):]
    if CKPT_KEY_PREFIX and k.startswith(CKPT_KEY_PREFIX): k = k[len(CKPT_KEY_PREFIX):]
    return k.replace(".LayerNorm.gamma", ".LayerNorm.weight").replace(".LayerNorm.beta", ".LayerNorm.bias")

def split_bert_checkpoint(raw):
    sd = raw
    if isinstance(raw, dict):
        for k in ("model_state_dict", "model", "state_dict", "global_state_dict", "theta_g"):
            if k in raw and isinstance(raw[k], dict): sd = raw[k]; break
    enc = {}
    for k, v in sd.items():
        if not torch.is_tensor(v): continue
        k = _norm_key(k)
        if k.startswith("cls."): continue
        enc[k[len("bert."):] if k.startswith("bert.") else k] = v
    return enc

def make_student(pre, layer_map):
    """Half-depth copy of a loaded backbone: embeddings / norms copied, layer i <- pre-trained layer layer_map[i]."""
    sd = {k: v.detach().cpu().clone() for k, v in pre.state_dict().items()}
    if KIND == 'eff':
        st = EfficientLegalLM(VOCAB_SIZE, D_MODEL, len(layer_map), N_HEADS, N_KV_HEADS, HEAD_DIM, D_FF, MAX_POS, ROPE_BASE, 0.0, WINDOW_SIZE); lp = 'layers.'
    else:
        st = BertModel(BertConfig(**{**M7_CFG, 'num_hidden_layers': len(layer_map)}), add_pooling_layer=False); lp = 'encoder.layer.'
    new = {k: v for k, v in sd.items() if not k.startswith(lp)}
    for i, src in enumerate(layer_map):
        pref = f'{lp}{src}.'
        for k, v in sd.items():
            if k.startswith(pref): new[f'{lp}{i}.' + k[len(pref):]] = v
    missing, unexpected = st.load_state_dict(new, strict=False)
    missing = [k for k in missing if k not in ('lm_head.weight', 'embeddings.position_ids')]
    assert not missing and not unexpected, (missing[:4], unexpected[:4])
    if KIND == 'eff':
        for i, src in enumerate(layer_map): st.layers[i].attn.use_window = (src % 2 == 0)   # same attention type as source layer
    return st

def load_backbone(layer_map=None):
    """Fresh backbone with the pre-trained weights of the chosen ARCH loaded (on CPU)."""
    ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
    if KIND == 'eff':
        bb = EfficientLegalLM(VOCAB_SIZE, D_MODEL, N_LAYERS, N_HEADS, N_KV_HEADS, HEAD_DIM, D_FF, MAX_POS, ROPE_BASE, 0.0, WINDOW_SIZE)
        state = {(k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v for k, v in ckpt["model_state_dict"].items()}
        missing, unexpected = bb.load_state_dict(state, strict=False)
        missing = [k for k in missing if k != "lm_head.weight"]
        assert not missing and not unexpected, f"checkpoint mismatch: missing={missing[:5]} unexpected={unexpected[:5]}"
    else:
        bb = BertModel(BertConfig(**M7_CFG), add_pooling_layer=False)
        enc = split_bert_checkpoint(ckpt)
        missing, unexpected = bb.load_state_dict(enc, strict=False)
        unexpected = [k for k in unexpected if k != "embeddings.position_ids" and not k.startswith("pooler.")]
        assert not missing, f"checkpoint mismatch - missing: {missing[:6]} unexpected: {unexpected[:6]} (check CKPT_KEY_PREFIX)"
    del ckpt
    if layer_map is not None:
        bb = make_student(bb, layer_map)
    return bb

class Encoder(nn.Module):
    """Window encoder: (N, L) ids + (N, L) bool mask -> (N, hidden) masked-mean vector (float32)."""
    def __init__(self, backbone, use_gc):
        super().__init__()
        self.backbone = backbone; self.use_gc = use_gc
        if KIND == 'bert' and use_gc:
            backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    def forward(self, ids, mask):
        m = self.backbone; T = ids.size(1)
        if KIND == 'eff':
            x = m.tok_emb(ids); cos, sin = m.freqs_cos[:T], m.freqs_sin[:T]
            full, win = build_masks(T, mask, BIDIR_WINDOW)
            for blk in m.layers:
                mk = win if blk.attn.use_window else full
                if self.use_gc and self.training:
                    x = grad_checkpoint(block_forward, blk, x, cos, sin, mk, use_reentrant=False)
                else:
                    x = block_forward(blk, x, cos, sin, mk)
            h = m.final_norm(x).float()
        else:
            h = m(input_ids=ids, attention_mask=mask.long(), token_type_ids=torch.zeros_like(ids)).last_hidden_state.float()
        am = mask.float().unsqueeze(-1)
        return (h * am).sum(1) / am.sum(1).clamp(min=1)

# ---- quick load sanity (no training) ----
_bb = load_backbone()
print(f'{ARCH}: loaded {CKPT_PATH} | params {sum(p.numel() for p in _bb.parameters())/1e6:.1f}M')
if KIND == 'eff':
    try:
        _bb = _bb.to(DEVICE).eval()
        _ids = torch.tensor([tokenizer(train[0]['text'], add_special_tokens=False)['input_ids'][:512]], device=DEVICE)
        with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=AMP):
            x = _bb.tok_emb(_ids); cs, sn = _bb.freqs_cos[:_ids.size(1)], _bb.freqs_sin[:_ids.size(1)]
            cm_ = torch.tril(torch.ones(_ids.size(1), _ids.size(1), dtype=torch.bool, device=DEVICE))[None, None]
            wm_ = cm_ & torch.triu(torch.ones_like(cm_), diagonal=-(WINDOW_SIZE - 1))
            for blk in _bb.layers: x = block_forward(blk, x, cs, sn, wm_ if blk.attn.use_window else cm_)
            lg = _bb.lm_head(_bb.final_norm(x))
        lm = F.cross_entropy(lg[0, :-1].float(), _ids[0, 1:]).item()
        print(f'causal LM sanity loss {lm:.3f} (random weights ~ {math.log(VOCAB_SIZE):.1f}; trained roughly 2-4)')
        assert lm < 6.0, 'LM loss too high - weights or tokenizer do not match'
    except AssertionError: raise
    except Exception as e: print('sanity check skipped:', repr(e)[:200])
del _bb; gc.collect()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()

## 5b. Low-rank adapters and the multi-label hierarchical model

In [ ]:
class LoRALinear(nn.Module):
    # y = Wx + alpha/r * B(Ax)
    def __init__(self,base,rank,alpha):
        super().__init__(); self.base=base
        self.A=nn.Parameter(torch.empty(rank,base.in_features)); self.B=nn.Parameter(torch.zeros(base.out_features,rank))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5)); self.scale=alpha/rank
    def forward(self,x):
        return self.base(x)+self.scale*F.linear(F.linear(x,self.A.to(x.dtype)),self.B.to(x.dtype))

class LoRMALinear(nn.Module):
    # multiplicative output-space adaptation: (I + alpha/r * B A) W x, W frozen, B starts at zero
    def __init__(self,base,rank,alpha):
        super().__init__(); self.base=base
        self.A=nn.Parameter(torch.randn(rank,base.out_features)*0.02); self.B=nn.Parameter(torch.zeros(base.out_features,rank))
        self.scale=alpha/rank
    def forward(self,x):
        h=self.base(x); return h+self.scale*F.linear(F.linear(h,self.A.to(h.dtype)),self.B.to(h.dtype))

def add_adapters(backbone,method):
    assert method in ('lora','lorma')
    wrapper=LoRALinear if method=='lora' else LoRMALinear
    layers = backbone.layers if KIND=='eff' else backbone.encoder.layer
    count=0
    for blk in layers:
        # every nn.Linear inside the layer (EffLM: q,k,v,out + gate,up,down | BERT: q,k,v,attn-out,ffn-up,ffn-down)
        targets=[(parent,name,child) for parent in blk.modules() for name,child in parent.named_children() if isinstance(child,nn.Linear)]
        for parent,name,child in targets:
            setattr(parent,name,wrapper(child,RANK,ALPHA)); count+=1
    return count

# Prior log-odds bias: start every statute at its base rate.
_p=(Y_train.sum(0)+0.5)/(len(Y_train)+1.0)
PRIOR_LOGIT=torch.tensor(np.log(_p/(1-_p)),dtype=torch.float32)

class LSIClassifier(nn.Module):
    def __init__(self,encoder,pooling,num_labels):
        super().__init__(); self.encoder=encoder; self.pooling=pooling; self.num_labels=num_labels
        h=HID; self.h=h; self.drop=nn.Dropout(DROPOUT)
        if pooling=='label_attention':
            self.label_q=nn.Parameter(torch.randn(num_labels,h)*0.02)
            self.label_w=nn.Parameter(torch.randn(num_labels,h)*0.02)
            self.label_b=nn.Parameter(PRIOR_LOGIT.clone())
        else:
            if pooling=='attention':
                self.chunk_att=nn.Sequential(nn.Linear(h,h//2),nn.Tanh(),nn.Linear(h//2,1))
            self.head=nn.Linear(h,num_labels)
            with torch.no_grad(): self.head.bias.copy_(PRIOR_LOGIT)
    def forward(self,ids,mask,cm,chunk_selection=None):
        B,C,L=ids.shape; valid=cm.reshape(-1)
        xi=ids.reshape(B*C,L)[valid]; xm=mask.reshape(B*C,L)[valid].bool()
        hs=self.encoder(xi,xm)
        emb=hs.new_zeros(B*C,hs.size(-1)); emb[valid]=hs; emb=emb.reshape(B,C,-1)
        allowed=cm.clone()
        if chunk_selection in ('first','middle','last'):
            allowed[:]=False
            for b in range(B):
                idx=torch.where(cm[b])[0]
                if len(idx):
                    p={'first':0,'middle':len(idx)//2,'last':-1}[chunk_selection]
                    allowed[b,idx[p]]=True
        if self.pooling=='label_attention':
            e=emb.float()
            sc=torch.einsum('bch,kh->bck',e,self.label_q.float())/math.sqrt(self.h)   # (B, windows, labels)
            sc=sc.masked_fill(~allowed.unsqueeze(-1),-1e4)
            w=torch.softmax(sc,dim=1)                                                 # softmax over windows
            doc=self.drop(torch.einsum('bck,bch->bkh',w,e))                            # (B, labels, h)
            logits=(doc*self.label_w.float()).sum(-1)+self.label_b.float()
            return logits,w.mean(-1)
        if self.pooling=='mean':
            w=allowed.float()/allowed.sum(-1,keepdim=True).clamp(min=1)
        else:
            sc=self.chunk_att(emb).squeeze(-1).float().masked_fill(~allowed,-1e4)
            w=torch.softmax(sc,dim=-1)
        doc=(emb.float()*w.unsqueeze(-1)).sum(dim=1)
        return self.head(self.drop(doc)).float(),w

def make_model(method):
    """method: 'full' | 'lora' | 'lorma' | 'kd' (kd = half-depth student, all weights trainable)."""
    bb=load_backbone(KD_LAYER_MAP if method=='kd' else None)
    if method in ('lora','lorma'):
        for p in bb.parameters(): p.requires_grad_(False)
        print(method,'adapted linear layers',add_adapters(bb,method))
    else:
        for p in bb.parameters(): p.requires_grad_(True)
    enc=Encoder(bb,use_gc=(GRAD_CHECKPOINT and method in ('full','kd')))
    model=LSIClassifier(enc,POOLING,NUM_LABELS).to(DEVICE)
    print(ARCH,method,'trainable / total:',sum(p.numel() for p in model.parameters() if p.requires_grad),'/',sum(p.numel() for p in model.parameters()))
    return model

## 6. Loss functions and threshold utilities (NumPy only)

In [ ]:
def f1_stats(Y,Pred):
    Y=np.asarray(Y,dtype=np.float64);Pred=np.asarray(Pred,dtype=np.float64)
    tp=(Y*Pred).sum(0);fp=((1-Y)*Pred).sum(0);fn=(Y*(1-Pred)).sum(0)
    denom=2*tp+fp+fn;f1=np.where(denom>0,2*tp/np.maximum(denom,1),0.0)
    present=Y.sum(0)>0
    prec=np.where(tp+fp>0,tp/np.maximum(tp+fp,1),0.0);rec=np.where(tp+fn>0,tp/np.maximum(tp+fn,1),0.0)
    micro=2*tp.sum()/max(2*tp.sum()+fp.sum()+fn.sum(),1)
    return {'macro_f1':float(f1[present].mean()) if present.any() else 0.0,'macro_f1_all':float(f1.mean()),
            'micro_f1':float(micro),'subset_acc':float((Y==Pred).all(1).mean()),
            'macro_precision':float(prec[present].mean()) if present.any() else 0.0,
            'macro_recall':float(rec[present].mean()) if present.any() else 0.0,
            'avg_pred_labels':float(Pred.sum(1).mean())}

def decode(P,thr,min_pred=MIN_PRED):
    thr=np.broadcast_to(np.asarray(thr,dtype=np.float64),(P.shape[1],))
    pred=(P>=thr[None,:]).astype(np.float32)
    if min_pred>0:
        ratio=P/np.maximum(thr[None,:],1e-9)
        for i in np.where(pred.sum(1)<min_pred)[0]:
            pred[i,np.argsort(-ratio[i])[:min_pred-pred[i].sum().astype(int)]]=1
    return pred

THR_GRID=np.round(np.arange(0.03,0.96,0.02),3)

def _f1_per_label(Y,pred):
    tp=(Y*pred).sum(0);fp=((1-Y)*pred).sum(0);fn=(Y*(1-pred)).sum(0);d=2*tp+fp+fn
    return np.where(d>0,2*tp/np.maximum(d,1),0.0)

def best_global_threshold(Y,P,grid=THR_GRID):
    present=Y.sum(0)>0
    scores=[_f1_per_label(Y,(P>=t).astype(np.float64))[present].mean() for t in grid]
    j=int(np.argmax(np.round(scores,6)-1e-6*np.abs(grid-0.5)));return float(grid[j]),float(scores[j])

def per_label_thresholds(Y,P,g,min_support=MIN_SUPPORT_PL,shrink=SHRINK_PL,grid=THR_GRID):
    F_=np.stack([_f1_per_label(Y,(P>=t).astype(np.float64)) for t in grid])
    F_=F_-1e-6*np.abs(grid-g)[:,None]
    best=grid[np.argmax(F_,axis=0)]
    thr=np.full(Y.shape[1],g,dtype=np.float64);ok=Y.sum(0)>=min_support
    thr[ok]=shrink*best[ok]+(1-shrink)*g
    return thr

def cv_compare(Y,P,repeats=5,seed=0):
    rng=np.random.RandomState(seed);res={'global':[],'per_label':[]}
    for _ in range(repeats):
        perm=rng.permutation(len(Y));halves=[perm[:len(Y)//2],perm[len(Y)//2:]]
        for a,b in (halves,halves[::-1]):
            g,_s=best_global_threshold(Y[a],P[a]);pl=per_label_thresholds(Y[a],P[a],g)
            res['global'].append(f1_stats(Y[b],decode(P[b],g))['macro_f1'])
            res['per_label'].append(f1_stats(Y[b],decode(P[b],pl))['macro_f1'])
    return {k:float(np.mean(v)) for k,v in res.items()}

def choose_thresholds(Y,P,mode=THRESHOLD_MODE):
    g,gs=best_global_threshold(Y,P);pl=per_label_thresholds(Y,P,g);cv=cv_compare(Y,P)
    use='per_label' if (mode=='per_label' or (mode=='auto' and cv['per_label']>cv['global']+1e-4)) else 'global'
    thr=pl if use=='per_label' else np.full(Y.shape[1],g)
    rep={'global_threshold':g,'mode_used':use,'cv_macro_f1':cv,
         'dev_global':f1_stats(Y,decode(P,g)),'dev_per_label_in_sample':f1_stats(Y,decode(P,pl)),'dev_final':f1_stats(Y,decode(P,thr))}
    return thr,rep

def build_pos_weight(Y):
    pos=Y.sum(0);neg=len(Y)-pos
    pw=((neg+1)/(pos+1))**POS_WEIGHT_POWER
    return np.clip(pw,1.0,POS_WEIGHT_MAX).astype(np.float32)

def make_loss(Y):
    pw=torch.tensor(build_pos_weight(Y),device=DEVICE)
    def loss_fn(logits,y):
        logits=logits.float()
        if LOSS=='bce':
            return F.binary_cross_entropy_with_logits(logits,y,pos_weight=pw)
        if LOSS=='focal':
            bce=F.binary_cross_entropy_with_logits(logits,y,reduction='none',pos_weight=pw)
            p=torch.sigmoid(logits);pt=y*p+(1-y)*(1-p)
            return (((1-pt).clamp(min=0)**FOCAL_GAMMA)*bce).mean()
        p=torch.sigmoid(logits);xn=(1-p+ASL_CLIP).clamp(max=1)
        lp=y*torch.log(p.clamp(min=1e-8));ln=(1-y)*torch.log(xn.clamp(min=1e-8))
        pt=p*y+xn*(1-y);gam=ASL_GAMMA_POS*y+ASL_GAMMA_NEG*(1-y)
        return -(((1-pt).clamp(min=0)**gam)*(lp+ln)).mean()
    return loss_fn

_rs=np.random.RandomState(0);_Y=(_rs.rand(200,12)<0.15).astype(np.float32);_P=np.clip(_Y*0.55+_rs.rand(200,12)*0.5,0,1)
_g,_=best_global_threshold(_Y,_P);_pred=decode(_P,_g)
assert _pred.sum(1).min()>=MIN_PRED and set(np.unique(_pred))<={0.0,1.0}
print('threshold utils OK | synthetic global thr',_g,'| f1',round(f1_stats(_Y,_pred)['macro_f1'],3))

## 7. Training, document-level evaluation, checkpoint selection

In [ ]:
def loader(ds,shuffle=False,batch=None):
    return DataLoader(ds,batch_size=batch or BATCH_SIZE,shuffle=shuffle,collate_fn=collate,num_workers=NUM_WORKERS)

def move(batch):
    ids,mask,cm,y,keys=batch
    return ids.to(DEVICE),mask.to(DEVICE),cm.to(DEVICE),y.to(DEVICE),keys

@torch.no_grad()
def predict_prob(model,ds,selection=None,as_logits=False):
    model.eval();probs=[];ys=[];all_ids=[]
    for b in tqdm(loader(ds,batch=INFER_BATCH),desc='Inference',leave=False):
        x,mask,cm,y,keys=move(b)
        with torch.autocast(device_type='cuda',dtype=AMP_DTYPE,enabled=AMP):
            logit,_=model(x,mask,cm,selection)
        probs.append((logit.float() if as_logits else torch.sigmoid(logit.float())).cpu().numpy());ys.append(y.cpu().numpy());all_ids.extend(keys)
    return np.concatenate(probs),np.concatenate(ys),all_ids

def _is_adapter_key(k): return k.endswith('.A') or k.endswith('.B')

def reload_best(model,path,method=None):
    payload=torch.load(path,map_location='cpu',weights_only=False)
    assert payload['label_ids']==LABEL_IDS,'label space changed since checkpoint was saved'
    assert payload.get('arch',ARCH)==ARCH, f"checkpoint arch {payload.get('arch')} != {ARCH}"
    if method is not None: assert payload['method']==method,f"checkpoint method {payload['method']} != requested {method}"
    missing,unexpected=model.load_state_dict(payload['state'],strict=False)
    assert not unexpected,unexpected
    if payload['method'] in ('full','kd'): assert not missing,missing[:15]
    else: assert all(k.startswith('encoder.') and not _is_adapter_key(k) for k in missing),missing[:15]
    print('loaded',path,'| saved best dev macro-F1:',payload.get('best_dev_macro_f1'))
    return model

TEACHER_INFO={}

def fit_core(model, method, lr, path, desc, kd_logits=None, kd_index=None):
    """Early-stopped training on dev macro-F1. method = tag stored in checkpoint ('full'|'lora'|'lorma'|'kd')."""
    g_enc=[p for n,p in model.named_parameters() if p.requires_grad and n.startswith('encoder.')]
    g_head=[p for n,p in model.named_parameters() if p.requires_grad and not n.startswith('encoder.')]
    groups=[{'params':g_head,'lr':HEAD_LR}]+([{'params':g_enc,'lr':lr}] if g_enc else [])
    opt=torch.optim.AdamW(groups,weight_decay=WEIGHT_DECAY)
    total_steps=math.ceil(len(loader(train_ds,True))/GRAD_ACCUM)*EPOCHS
    sched=get_linear_schedule_with_warmup(opt,int(total_steps*WARMUP_FRAC),total_steps)
    scaler=torch.amp.GradScaler('cuda',enabled=AMP)
    loss_fn=make_loss(Y_train)
    best=-1;bad=0
    clip_params=[p for p in model.parameters() if p.requires_grad]
    for epoch in range(EPOCHS):
        model.train();opt.zero_grad(set_to_none=True);losses=[]
        batches=loader(train_ds,shuffle=True)
        for i,batch in enumerate(tqdm(batches,desc=f'{ARCH} {desc} epoch {epoch+1}/{EPOCHS}',leave=False)):
            x,am,cm,y,keys=move(batch)
            with torch.autocast(device_type='cuda',dtype=AMP_DTYPE,enabled=AMP):
                logit,_=model(x,am,cm)
            loss=loss_fn(logit,y)
            if kd_logits is not None:      # distillation: blend task loss with soft teacher targets
                tl=torch.from_numpy(kd_logits[[kd_index[k] for k in keys]]).to(DEVICE)
                kd=F.binary_cross_entropy_with_logits(logit.float()/KD_TEMP,torch.sigmoid(tl/KD_TEMP))*KD_TEMP*KD_TEMP
                loss=(1.0-KD_ALPHA)*loss+KD_ALPHA*kd
            group_size=min(GRAD_ACCUM,len(batches)-(i//GRAD_ACCUM)*GRAD_ACCUM)
            scaler.scale(loss/group_size).backward();losses.append(loss.item())
            if (i+1)%GRAD_ACCUM==0 or (i+1)==len(batches):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(clip_params,1.0)
                scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True);sched.step()
        probs,Yd,_=predict_prob(model,dev_ds)
        g,score=best_global_threshold(Yd,probs)
        m=f1_stats(Yd,decode(probs,g))
        print(f"{ARCH} {desc} epoch {epoch+1} train_loss {np.mean(losses):.4f} | dev thr {g:.2f} macroF1 {m['macro_f1']:.4f} microF1 {m['micro_f1']:.4f} subset {m['subset_acc']:.4f} avg#pred {m['avg_pred_labels']:.2f}")
        if m['macro_f1']>best+1e-6:
            best=m['macro_f1'];bad=0
            keep={k:v.detach().cpu().clone() for k,v in model.state_dict().items()
                  if method in ('full','kd') or not k.startswith('encoder.') or _is_adapter_key(k)}
            torch.save({'state':keep,'method':method,'arch':ARCH,'representation':REPRESENTATION,'pooling':POOLING,'loss':LOSS,
                        'label_ids':LABEL_IDS,'rank':RANK,'alpha':ALPHA,'kd_layer_map':KD_LAYER_MAP if method=='kd' else None,
                        'max_len':MAX_LEN,'chunks':CHUNKS,'stride':STRIDE,'best_dev_macro_f1':best},path)
        else:
            bad+=1
            if bad>=PATIENCE: print('Early stopping');break
    return model,path

def fit(method, init_ckpt=None):
    if method!='kd':
        model=make_model(method)
        if init_ckpt:
            model=reload_best(model,init_ckpt,method).to(DEVICE); print('warm-started from',init_ckpt)
        return fit_core(model,method,FULL_LR if method=='full' else ADAPTER_LR,OUTPUT_DIR/f'best_{method}_lsi.pt',method)
    # ---------- stage 1: TEACHER = full-depth model, every weight fine-tuned ----------
    teacher=make_model('full')
    if KD_TEACHER_CKPT and os.path.exists(KD_TEACHER_CKPT):
        tpath=KD_TEACHER_CKPT; print('using teacher checkpoint',tpath)
    else:
        _,tpath=fit_core(teacher,'full',KD_TEACHER_LR,OUTPUT_DIR/'best_teacher_lsi.pt','teacher')
    teacher=reload_best(teacher,tpath,'full').to(DEVICE)
    pdv,Ydv,_=predict_prob(teacher,dev_ds); tg,_=best_global_threshold(Ydv,pdv)
    TEACHER_INFO.update(f1_stats(Ydv,decode(pdv,tg))); print(f"[teacher] dev macroF1 {TEACHER_INFO['macro_f1']:.4f} microF1 {TEACHER_INFO['micro_f1']:.4f}")
    cache=OUTPUT_DIR/f'teacher_train_logits_{ARCH}.npy'
    if cache.exists() and np.load(cache,mmap_mode='r').shape==(len(train),NUM_LABELS):
        TL=np.load(cache); print('loaded cached teacher logits',cache)
    else:
        TL,_,_=predict_prob(teacher,train_ds,as_logits=True); np.save(cache,TL)    # soft labels for every training case
    del teacher; gc.collect()
    if DEVICE.type=='cuda': torch.cuda.empty_cache()
    # ---------- stage 2: STUDENT = half the layers, init from evenly spaced pre-trained layers ----------
    print(f'[student] {len(KD_LAYER_MAP)} layers copied from pre-trained layers {KD_LAYER_MAP}')
    student=make_model('kd')
    idx={r['id']:i for i,r in enumerate(train)}
    return fit_core(student,'kd',FULL_LR,OUTPUT_DIR/'best_kd_lsi.pt','student',kd_logits=TL,kd_index=idx)

## 8. Run the chosen adaptation method(s); compare dev metrics and threshold strategies

In [ ]:
results={}; predictions={}; training_summary=[]
for method in RUN_METHODS:
    print('\n===',ARCH.upper(),method.upper(),'===')
    use_init = INIT_CKPT if (INIT_CKPT and os.path.exists(INIT_CKPT)) else None
    model,path=fit(method, use_init)
    model=reload_best(model,path,method).to(DEVICE)
    pdev,Yd,dev_ids=predict_prob(model,dev_ds)
    ptest,_,test_ids=predict_prob(model,test_ds)
    assert test_ids==[r['id'] for r in test] and dev_ids==[r['id'] for r in dev]
    thr,rep=choose_thresholds(Yd,pdev)
    res={'arch':ARCH,'method':method,'representation':REPRESENTATION,'pooling':POOLING,'loss':LOSS,**rep,
         'teacher_dev':(dict(TEACHER_INFO) if method=='kd' else None),'dev_macro_f1':rep['dev_final']['macro_f1'],'dev_micro_f1':rep['dev_final']['micro_f1'],'dev_subset_acc':rep['dev_final']['subset_acc']}
    results[method]=res
    predictions[method]={'dev_prob':pdev,'test_prob':ptest,'dev_ids':dev_ids,'test_ids':test_ids,'thr':thr}
    np.save(OUTPUT_DIR/f'dev_prob_{method}.npy',pdev);np.save(OUTPUT_DIR/f'test_prob_{method}.npy',ptest)
    print(f"global thr {rep['global_threshold']:.2f} | CV macro-F1 (held-out halves): global {rep['cv_macro_f1']['global']:.4f} vs per-label {rep['cv_macro_f1']['per_label']:.4f} -> using {rep['mode_used']}")
    print('dev global     :',{k:round(v,4) for k,v in rep['dev_global'].items()})
    print('dev per-label* :',{k:round(v,4) for k,v in rep['dev_per_label_in_sample'].items()},'(*in-sample, optimistic)')
    print('dev FINAL      :',{k:round(v,4) for k,v in rep['dev_final'].items()})
    if method=='kd': print(f"KD: teacher dev macroF1 {TEACHER_INFO.get('macro_f1',float('nan')):.4f} -> student {res['dev_macro_f1']:.4f}")
    training_summary.append(res)
    del model;gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
print('\nRanking by dev Macro-F1:')
for row in sorted(training_summary,key=lambda r:r['dev_macro_f1'],reverse=True):
    print(f"  {row['arch']} {row['method']:6s} macroF1 {row['dev_macro_f1']:.4f} microF1 {row['dev_micro_f1']:.4f} subset {row['dev_subset_acc']:.4f} thresholds={row['mode_used']}")
with open(OUTPUT_DIR/'comparison.json','w') as f:json.dump(training_summary,f,indent=2)

## 9. Diagnostics: rare statutes and window contribution (dev-only)

In [ ]:
winner=max(results,key=lambda k:results[k]['dev_macro_f1'])
pr=predictions[winner];Pd=pr['dev_prob'];thr=pr['thr'];Pred=decode(Pd,thr)
f1l=_f1_per_label(Y_dev.astype(np.float64),Pred.astype(np.float64));support=Y_dev.sum(0);trp=Y_train.sum(0)
for lo,hi in [(0,0),(1,5),(6,20),(21,100),(101,10**9)]:
    sel=(trp>=lo)&(trp<=hi)&(support>0)
    if sel.any(): print(f"train freq {lo:>3}-{hi if hi<10**9 else 'inf':<4} | {int(sel.sum()):4d} labels in dev | mean F1 {f1l[sel].mean():.4f}")
print('Statutes ever predicted on test:',int((decode(pr['test_prob'],thr).sum(0)>0).sum()),'/',NUM_LABELS)

In [ ]:
model=make_model(winner);model=reload_best(model,OUTPUT_DIR/f'best_{winner}_lsi.pt').to(DEVICE)
ablation=[]
for selection in (None,'first','middle','last'):
    pp,yy,_=predict_prob(model,dev_ds,selection)
    row={'chunks_used':selection or 'all',**f1_stats(yy,decode(pp,thr))}
    ablation.append(row);print({k:(round(v,4) if isinstance(v,float) else v) for k,v in row.items()})
with open(OUTPUT_DIR/'chunk_ablation.json','w') as f:json.dump(ablation,f,indent=2)
del model;gc.collect()
if DEVICE.type=='cuda':torch.cuda.empty_cache()

## 10. Create the exact competition JSON and ZIP
One `{id, pred}` per test document, in test order; `pred` = sorted unique int statute IDs, never empty.

In [ ]:
winner=max(results,key=lambda k:results[k]['dev_macro_f1'])
pr=predictions[winner];thr=pr['thr']
test_pred=decode(pr['test_prob'],thr)
VALID=set(LABEL_IDS)
submission=[{'id':str(case_id),'pred':sorted({int(LABEL_IDS[j]) for j in np.where(test_pred[i]>0)[0]})} for i,case_id in enumerate(pr['test_ids'])]
assert len(submission)==len(test)==len({r['id'] for r in submission})
assert [r['id'] for r in submission]==[r['id'] for r in test]
for r in submission:
    assert set(r)=={'id','pred'} and isinstance(r['pred'],list) and len(r['pred'])>=1
    assert all(type(v) is int and v in VALID for v in r['pred']) and len(r['pred'])==len(set(r['pred']))
output=OUTPUT_DIR/'prediction.json'
with open(output,'w',encoding='utf-8') as f:json.dump(submission,f,ensure_ascii=False,indent=2)
zip_path=OUTPUT_DIR/f'submission_lsi_{ARCH}_{winner}.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:z.write(output,arcname='prediction.json')
_pd=decode(pr['dev_prob'],thr);_pres=Y_dev.sum(0)>0
print('dev macro-F1 mine vs sklearn:',round(f1_stats(Y_dev,_pd)['macro_f1'],4),round(f1_score(Y_dev[:,_pres],_pd[:,_pres],average='macro',zero_division=0),4))
print('arch:',ARCH,'| winner method:',winner,'| thresholds:',results[winner]['mode_used'],'| test size:',len(submission),'| avg #statutes/doc:',round(np.mean([len(r['pred']) for r in submission]),2))
print('saved:',output,zip_path,'sample:',submission[:3])
try:
    from IPython.display import display,FileLink
    display(FileLink(str(output)));display(FileLink(str(zip_path)))
except Exception:pass

## 11. Suggested controlled experiments
Change **one** switch in Section 2 per run (same seed/splits): `ARCH` (`model4` / `model6` / `model7`), `RUN_METHODS` (`full` / `lora` / `lorma` / `kd`), `REPRESENTATION`, `POOLING`, `LOSS`, `THRESHOLD_MODE`. Each run writes to its own folder `lsi_p5_<ARCH>`; the saved `dev_prob_*.npy` / `test_prob_*.npy` allow averaging models afterwards.

**KD cost:** `kd` trains a full teacher first (or reuses `KD_TEACHER_CKPT`, a `best_*_lsi.pt` of a *full* run of the same ARCH), runs one inference pass over the training set for soft labels (cached), then trains the half-depth student.

**Memory (T4):** Model 6 (1024d, 20 layers) with `full` is the heaviest; gradient checkpointing is on by default. If it OOMs, lower `CHUNKS` / `MAX_LEN` or use `lora`/`lorma`. `label_attention` adds ~2×K×hidden parameters.